# `fastcdp` Route Interception & Dialog Harness

> Reusable `fastcdp` async context manager (`open_solveit`), CDP `Fetch` HTML/asset interceptor (`install_craft_routes`), health probe (`inspect_dialog`), and smoke checks (`smoke_dialog`) for testing local CRAFT changes safely against any Solveit dialog.

In [ ]:
#| default_exp cdp

In [ ]:
#| export
import asyncio, base64, json, mimetypes, tempfile
from contextlib import AsyncExitStack, asynccontextmanager
from dataclasses import dataclass, field
from urllib.parse import unquote, urlparse
from fastcore.meta import delegates
from fastcore.utils import *
import fast5ever
from fast5ever import Element, Text

from craft_dev.auth import (
    DEFAULT_AUTH_FILE, REPO_ROOT, alogin, cookies_for_cdp, default_target_url,
    ensure_auth, is_authenticated_page, resolve_browser_executable,)
from craft_dev.build import CRAFT_ROOT_ASSETS

In [ ]:
#| export
ROOT_ASSET_NAMES = set(CRAFT_ROOT_ASSETS)
_CONTENT_TYPES = {'.js': 'application/javascript; charset=utf-8', '.css': 'text/css; charset=utf-8', '.svg': 'image/svg+xml'}
_CORE_GLOBALS = ('craftC2', 'craftOmit', 'craftParts', 'craftActionMenus', 'craftCellChains', 'solveitHotkeys')
DESKTOP_METRICS = dict(width=1440, height=900, deviceScaleFactor=1, mobile=False)
MOBILE_METRICS = dict(width=390, height=844, deviceScaleFactor=3, mobile=True)

@dataclass
class InterceptRecord:
    url: str
    action: str  # "served-local" | "blocked-vanilla" | "passthrough"
    local_path: str | None = None

@dataclass
class InspectReport:
    url: str
    final_url: str
    mode: str
    assets_dir: str
    authenticated: bool
    title: str
    intercepted: list[InterceptRecord] = field(default_factory=list)
    console_errors: list[str] = field(default_factory=list)
    console_warnings: list[str] = field(default_factory=list)
    page_errors: list[str] = field(default_factory=list)
    craft_health: dict = field(default_factory=dict)
    eval_result: object = None
    screenshot: str | None = None

In [ ]:
#| export
HEALTH_PROBE_JS = r"""(() => {
  const globals = [
    'craftC2', 'craftMd', 'craftOmit', 'craftParts', 'craftActionMenus', 'craftCellChains',
    'craftTables', 'craftNoMonaco', 'craftInsertPreview', 'craftDesktopUX', 'craftImageDrop',
    'craftMobile', 'craftMobileBar', 'craftContext', 'craftFonts', 'craftStruct', 'craftNavRef',
    'craftHistory', 'craftTTS', 'craftBugCapture', 'solveitHotkeys', 'solveitFileTree', '__hotkeyRegister',
  ];
  const globalStatus = {};
  for (const name of globals) {
    const val = window[name];
    globalStatus[name] = val == null ? null : typeof val;
  }
  const cells = [...document.querySelectorAll('#dialog-container > .editable')];
  const cellCounts = {};
  for (const c of cells) {
    const mtype = c.querySelector('.card-in')?.dataset?.mtype || c.dataset?.mtype || 'unknown';
    cellCounts[mtype] = (cellCounts[mtype] || 0) + 1;
  }
  const cs = getComputedStyle(document.documentElement);
  const cssVars = {
    '--craft-prose': cs.getPropertyValue('--craft-prose').trim(),
    '--craft-code': cs.getPropertyValue('--craft-code').trim(),
    '--craft-prose-family': cs.getPropertyValue('--craft-prose-family').trim(),
    '--craft-code-family': cs.getPropertyValue('--craft-code-family').trim(),
  };
  const craftTags = [...document.querySelectorAll('script[src], link[href]')]
    .map(el => el.getAttribute('src') || el.getAttribute('href'))
    .filter(u => u && (/CRAFT\.(js|css)/i.test(u) || u.includes('.craft-assets')));
  return {
    hasDialogContainer: !!document.getElementById('dialog-container'),
    hasFileTreePanel: !!document.getElementById('solveit-file-tree-panel'),
    hasLauncher: !!document.getElementById('solveit-launcher'),
    isMobile: document.documentElement.classList.contains('craft-mobile'),
    totalCells: cells.length, cellCounts,
    hotkeyCount: window.solveitHotkeys?.list?.()?.length ?? null,
    globals: globalStatus, cssVars, craftTags,
  };
})()"""

## Local Asset Resolution & Inline HTML Patching

Solveit serves `.craft-assets/*` as static files and **inlines** `CRAFT.css` (`<style>`) and `CRAFT.js` (`<script type="module">`) directly into `/dialog_` HTML responses. `install_craft_routes` intercepts both using the CDP `Fetch` domain.

In [ ]:
#| export
def resolve_local_asset(url_path, assets_dir):
    "Resolve a Solveit asset `url_path` to a local file in `assets_dir`, if present."
    raw = unquote(url_path.split('?', 1)[0])
    parts = [p for p in raw.split('/') if p]
    if not parts or any(p in ('.', '..') or '\\' in p for p in parts): return None
    base = Path(assets_dir).resolve()
    if '.craft-assets' in parts:
        tail = parts[parts.index('.craft-assets') + 1:]
        if not tail: return None
        rel = Path(*tail)
        return first((c for c in (base / '.craft-assets' / rel, base / rel) if c.is_file() and c.resolve().is_relative_to(base)))
    if parts[-1] in ROOT_ASSET_NAMES and (c := base/parts[-1]).is_file() and c.resolve().is_relative_to(base): return c
    return None

def guess_content_type(path):
    "Return the HTTP `Content-Type` header for local asset `path`."
    return _CONTENT_TYPES.get(path.suffix.lower()) or mimetypes.guess_type(str(path))[0] or 'application/octet-stream' 

In [ ]:
import tempfile
from fastcore.test import test_eq

`resolve_local_asset` maps both root `CRAFT.js`/`CRAFT.css` requests and `.craft-assets/*` paths to local files, ignoring query strings.

In [ ]:
test_eq(resolve_local_asset('/static/CRAFT.js?v=1', REPO_ROOT), REPO_ROOT/'CRAFT.js')
test_eq(resolve_local_asset('/static/.craft-assets/craft_md.js', REPO_ROOT), REPO_ROOT/'craft_md.js')
test_eq(resolve_local_asset('/static/unrelated.js', REPO_ROOT), None)
test_eq(guess_content_type(Path('CRAFT.js')), 'application/javascript; charset=utf-8')
test_eq(resolve_local_asset('/static/.craft-assets/../../pyproject.toml', REPO_ROOT), None)
test_eq(resolve_local_asset('/static/.craft-assets/%2e%2e/pyproject.toml', REPO_ROOT), None)
test_eq(resolve_local_asset('/static/.craft-assets/', REPO_ROOT), None)

In [ ]:
#| export
def _iter_elements(node):
    for child in list(node.element_children):
        yield child
        yield from _iter_elements(child)

def _patch_block(doc, tag, asset_name, pred, path, label, parent_tag, attrs, mode, records):
    replaced = False
    for el in _iter_elements(doc):
        if not el.is_tag(tag): continue
        if el.attrs.get('data-craft-asset') != asset_name and not pred(el.to_text()): continue
        replaced = True
        if mode == 'vanilla':
            records.append(InterceptRecord(url=label, action='blocked-vanilla'))
            el.detach()
        elif path.is_file():
            records.append(InterceptRecord(url=label, action='served-local', local_path=str(path)))
            el.attrs['data-craft-asset'] = asset_name
            for child in list(el.children): child.detach()
            el.append_child(Text(f"\n{path.read_text(encoding='utf-8')}\n"))
    if mode == 'local' and not replaced and path.is_file():
        parent = first(el for el in _iter_elements(doc) if el.is_tag(parent_tag))
        if parent is not None:
            records.append(InterceptRecord(url=f"{label} (injected)", action='served-local', local_path=str(path)))
            el = Element(tag, {**(attrs or {}), 'data-craft-asset': asset_name})
            el.append_child(Text(f"\n{path.read_text(encoding='utf-8')}\n"))
            parent.append_child(el)

def patch_dialog_html(html, *, mode, assets_dir):
    "Replace or strip Solveit's inlined `CRAFT.css` `<style>` and `CRAFT.js` `<script>` blocks."
    records = []
    if mode == 'live': return html, records
    doc = fast5ever.parse(html)
    _patch_block(doc, 'style', 'CRAFT.css', lambda b: '--craft-prose' in b or 'Tokyo Night Pure' in b or 'nbs/craft/' in b,
        assets_dir/'CRAFT.css', 'inline:<style>CRAFT.css', 'head', None, mode, records)
    _patch_block(doc, 'script', 'CRAFT.js', lambda b: ('craftC2' in b and 'hotkey_manager.js' in b) or '__craftCMCore' in b or 'nbs/craft/' in b,
        assets_dir/'CRAFT.js', 'inline:<script>CRAFT.js', 'body', {'type': 'module'}, mode, records)
    return doc.to_html(), records

In `local` mode `patch_dialog_html` swaps inlined CRAFT blocks with local files (injecting them if absent); in `vanilla` mode it strips them.

In [ ]:
with tempfile.TemporaryDirectory() as d:
    td = Path(d)
    (td/'CRAFT.css').write_text('/* LOCAL CSS */')
    (td/'CRAFT.js').write_text('/* LOCAL JS */')
    html = '<html><head><style>:root{--craft-prose:11px;}</style></head><body><script type="module">// hotkey_manager.js craftC2</script></body></html>'
    patched, recs = patch_dialog_html(html, mode='local', assets_dir=td)
    test_eq(('/* LOCAL CSS */' in patched, '/* LOCAL JS */' in patched, [r.action for r in recs]), (True, True, ['served-local', 'served-local']))
    vanilla, vrecs = patch_dialog_html(html, mode='vanilla', assets_dir=td)
    test_eq(('--craft-prose' in vanilla, 'craftC2' in vanilla, [r.action for r in vrecs]), (False, False, ['blocked-vanilla', 'blocked-vanilla']))
    tricky = '<html><head><!-- <style>--craft-prose</style> --></head><body><textarea><script>hotkey_manager.js craftC2</script></textarea></body></html>'
    inj, irecs = patch_dialog_html(tricky, mode='local', assets_dir=td)
    test_eq(('/* LOCAL CSS */' in inj, '/* LOCAL JS */' in inj, '<textarea>' in inj, [r.url for r in irecs]),
        (True, True, True, ['inline:<style>CRAFT.css (injected)', 'inline:<script>CRAFT.js (injected)']))

In [ ]:
#| export
_FETCH_PATTERNS = [dict(urlPattern='*/dialog*', resourceType='Document', requestStage='Response'),
    {'urlPattern': '*.craft-assets/*', 'requestStage': 'Request'}, {'urlPattern': '*/CRAFT.js*', 'requestStage': 'Request'},
    {'urlPattern': '*/CRAFT.css*', 'requestStage': 'Request'}]

async def _handle_paused(page, p, *, mode, assets_dir, records):
    rid, url = p['requestId'], p['request']['url']
    is_resp = 'responseStatusCode' in p
    try:
        if is_resp:
            code, hdrs = p['responseStatusCode'], p.get('responseHeaders') or []
            ctype = next((h['value'] for h in hdrs if h['name'].lower() == 'content-type'), '')
            if 200 <= code < 300 and 'text/html' in ctype.lower():
                res = await page.Fetch.getResponseBody(requestId=rid)
                raw = base64.b64decode(res['body']).decode('utf-8', 'replace') if res.get('base64Encoded') else res['body']
                patched, doc_recs = patch_dialog_html(raw, mode=mode, assets_dir=assets_dir)
                records.extend(doc_recs)
                out_hdrs = [h for h in hdrs if h['name'].lower() != 'content-length']
                b64 = base64.b64encode(patched.encode('utf-8')).decode('ascii')
                await page.Fetch.fulfillRequest(requestId=rid, responseCode=code, responseHeaders=out_hdrs, body=b64)
                return
            await page.Fetch.continueResponse(requestId=rid)
            return

        path = urlparse(url).path
        if mode == 'vanilla':
            ctype = 'text/css' if path.endswith('.css') else 'application/javascript'
            records.append(InterceptRecord(url=url, action='blocked-vanilla'))
            await page.Fetch.fulfillRequest(requestId=rid, responseCode=200,
                responseHeaders=[{'name': 'Content-Type', 'value': ctype}], body='')
            return
        if (local_file := resolve_local_asset(path, assets_dir)) is not None:
            records.append(InterceptRecord(url=url, action='served-local', local_path=str(local_file)))
            b64 = base64.b64encode(local_file.read_bytes()).decode('ascii')
            await page.Fetch.fulfillRequest(requestId=rid, responseCode=200,
                responseHeaders=[{'name': 'Content-Type', 'value': guess_content_type(local_file)}], body=b64)
            return
        records.append(InterceptRecord(url=url, action='passthrough'))
        await page.Fetch.continueRequest(requestId=rid)
    except Exception:
        try:
            if is_resp: await page.Fetch.continueResponse(requestId=rid)
            else: await page.Fetch.continueRequest(requestId=rid)
        except Exception: pass

@asynccontextmanager
async def install_craft_routes(page, *, mode='local', assets_dir=REPO_ROOT):
    "Enable CDP `Fetch` interception for CRAFT assets and inlined dialog `CRAFT.js`/`CRAFT.css`."
    records = []
    if mode == 'live':
        yield records
        return
    async with page.cdp.on('Fetch.requestPaused') as q:
        await page.Fetch.enable(patterns=_FETCH_PATTERNS)
        async def _pump():
            while True:
                m = await q.get()
                if m.get('sessionId') == page.sid:
                    await _handle_paused(page, m['params'], mode=mode, assets_dir=assets_dir, records=records)
        task = asyncio.create_task(_pump())
        try: yield records
        finally:
            task.cancel()
            try: await task
            except asyncio.CancelledError: pass
            try: await page.Fetch.disable()
            except Exception: pass

async def click_at(page, x, y):
    "Dispatch a mouse click at viewport coordinates `(x, y)`."
    for kind in ('mouseMoved', 'mousePressed', 'mouseReleased'):
        kw = dict(button='left', clickCount=1) if kind != 'mouseMoved' else {}
        await page.Input.dispatchMouseEvent(type=kind, x=x, y=y, **kw)

## Reusable Async Context Manager & High-Level Helpers

Use `async with open_solveit() as (page, intercepted, page_errors):` from any async test, script, or notebook in this repository to get an authenticated `fastcdp` `Page` with local CRAFT overrides already wired up.

In [ ]:
#| export
def default_assets_dir():
    "Return the default local CRAFT assets directory (`CRAFT_ASSETS_DIR`, `dist/`, or repo root)."
    if env := os.environ.get('CRAFT_ASSETS_DIR'): return Path(env).resolve()
    return (REPO_ROOT/'dist' if (REPO_ROOT/'dist'/'CRAFT.js').is_file() else REPO_ROOT).resolve()

def default_craft_mode():
    "Return default CRAFT asset mode from `CRAFT_TEST_MODE` or `'local'`."
    return os.environ.get('CRAFT_TEST_MODE', 'local')

def _fmt_exception(params):
    d = params.get('exceptionDetails') or {}
    return nested_idx(d, 'exception', 'description') or d.get('text') or str(d)

In [ ]:
#| export
@asynccontextmanager
async def _open_session(target_url, auth_path, *, mode, assets_dir, wait_ms, headed, exec_path, mobile):
    from fastcdp import CDP
    async with AsyncExitStack() as stack:
        td = stack.enter_context(tempfile.TemporaryDirectory(prefix='craft-cdp-'))
        cdp = await CDP.launch(user_data_dir=td, headless=not headed, reuse=False, chrome=exec_path)
        stack.push_async_callback(cdp.quit)
        if cookies := cookies_for_cdp(auth_path): await cdp.Storage.setCookies(cookies=cookies)
        page = await cdp.new_page()
        await page.Emulation.setDeviceMetricsOverride(**(MOBILE_METRICS if mobile else DESKTOP_METRICS))
        if mobile: await page.Emulation.setTouchEmulationEnabled(enabled=True)
        await page.start_console()
        page_errors = []
        exc_q = await stack.enter_async_context(cdp.on('Runtime.exceptionThrown'))
        async def _watch_exc():
            while True:
                m = await exc_q.get()
                if m.get('sessionId') == page.sid: page_errors.append(_fmt_exception(m['params']))
        exc_task = asyncio.create_task(_watch_exc())
        async def _cancel_exc():
            exc_task.cancel()
            try: await exc_task
            except asyncio.CancelledError: pass
        stack.push_async_callback(_cancel_exc)
        records = await stack.enter_async_context(install_craft_routes(page, mode=mode, assets_dir=assets_dir))
        await page.goto(target_url, wait='load', timeout=30)
        if wait_ms: await asyncio.sleep(wait_ms / 1000)
        yield page, records, page_errors

@asynccontextmanager
async def open_solveit(
    url=None, # Target Solveit dialog URL (defaults to `SOLVEIT_TEST_URL` or saved config)
    *,
    mode=None, # Asset mode: `'local'`, `'live'`, or `'vanilla'` (defaults to `CRAFT_TEST_MODE` or `'local'`)
    assets_dir=None, # Local CRAFT assets directory (defaults to `CRAFT_ASSETS_DIR`, `dist/`, or repo root)
    auth_file=DEFAULT_AUTH_FILE, # Saved cookie JSON path
    wait_ms=2000, # Milliseconds to wait after `load`
    headed=False, # Show browser window instead of headless mode
    browser_name='chrome', # `'chrome'`, `'chromium'`, or `'vivaldi'`
    executable_path=None, # Explicit browser binary path
    auto_auth=False, # Trigger Chrome login automatically if session is missing or expired
    mobile=False, # Emulate a mobile touch viewport (390×844)
):
    "Open an authenticated Solveit page via `fastcdp` with CRAFT route interception, yielding `(page, records, page_errors)`."
    target_url = url or default_target_url()
    if not target_url: raise ValueError("Pass url=... or run `craft-cdp auth --url ...` first to save a default URL.")
    auth_path = ensure_auth(auth_file, auto_login=auto_auth, url=target_url)
    exec_path = resolve_browser_executable(browser_name, executable_path)
    mode = mode or default_craft_mode()
    assets_dir = Path(assets_dir).resolve() if assets_dir else default_assets_dir()
    kw = dict(mode=mode, assets_dir=assets_dir, wait_ms=wait_ms, headed=headed, exec_path=exec_path, mobile=mobile)

    async with AsyncExitStack() as stack:
        page, records, page_errors = await stack.enter_async_context(_open_session(target_url, auth_path, **kw))
        if not await is_authenticated_page(page) and auto_auth:
            await stack.aclose()
            await alogin(url=target_url, auth_file=auth_path)
            page, records, page_errors = await stack.enter_async_context(_open_session(target_url, auth_path, **kw))
        if not await is_authenticated_page(page):
            cur = await page.eval('location.href')
            raise PermissionError(
                f"Session invalid or expired at {cur}. Run `.venv/bin/craft-cdp auth` to refresh.")
        yield page, records, page_errors

In [ ]:
#| export
async def ainspect_dialog(
    *, url=None, auth_file=DEFAULT_AUTH_FILE, mode='local', assets_dir=None, wait_ms=2000,
    eval_js=None, screenshot=None, headed=False, browser_name='chrome', executable_path=None, mobile=False,
):
    "Load a Solveit page via `fastcdp` and return a structured `InspectReport`."
    target_url = url or default_target_url()
    if not target_url: raise SystemExit("Pass --url or run `craft-cdp auth --url ...` first.")
    auth_path = ensure_auth(auth_file)
    exec_path = resolve_browser_executable(browser_name, executable_path)
    assets_dir = Path(assets_dir).resolve() if assets_dir else default_assets_dir()

    async with _open_session(target_url, auth_path, mode=mode, assets_dir=assets_dir,
        wait_ms=wait_ms, headed=headed, exec_path=exec_path, mobile=mobile) as (page, records, page_errors):
        final_url = await page.eval('location.href') or target_url
        authenticated = await is_authenticated_page(page)
        title = await page.eval('document.title') or ''
        health = await page.eval(HEALTH_PROBE_JS) if authenticated else {}
        eval_result = await page.eval(eval_js) if eval_js and authenticated else None
        msgs = await page.console()
        console_errors = [m for m in msgs if m.startswith('error:')]
        console_warnings = [m for m in msgs if m.startswith('warning:')]
        if screenshot:
            p = Path(screenshot)
            p.parent.mkdir(parents=True, exist_ok=True)
            p.write_bytes(await page.screenshot())

    return InspectReport(url=target_url, final_url=final_url, mode=mode, assets_dir=str(assets_dir), authenticated=authenticated,
        title=title, intercepted=records, console_errors=console_errors, console_warnings=console_warnings, page_errors=page_errors,
        craft_health=health, eval_result=eval_result, screenshot=str(screenshot) if screenshot else None)

@delegates(ainspect_dialog)
def inspect_dialog(**kwargs):
    "Synchronous wrapper around `ainspect_dialog`."
    return asyncio.run(ainspect_dialog(**kwargs))

In [ ]:
#| export
_OPTIONAL_SOLVEIT_SELS = frozenset({'promptCard', 'rawCard', 'cardOut', 'codeCard', 'noteCard', 'tblRow', 'ansiRed', 'skipActive', 'running'})

_FULL_EDITOR_HIDDEN_JS = "(() => !document.querySelector('#full_editor') && (window.monaco?.editor?.getEditors?.().length || 0) === 0 && (window.monaco?.editor?.getModels?.().length || 0) === 0)()"

_SOLVEIT_CONTRACT_JS = r"""(optionalKeys => {
  const selMap = window.__craftCMCore?.SOLVEIT?.sel;
  if (!selMap) return {ok: false, reason: 'CORE.SOLVEIT.sel missing'};
  const cells = [...document.querySelectorAll('#dialog-container > .editable')];
  const opt = new Set(optionalKeys);
  const matched = [], missingRequired = [], invalid = [];
  for (const [k, sel] of Object.entries(selMap)) {
    try {
      const hit = sel.trim().startsWith(':scope')
        ? cells.some(c => Boolean(c.querySelector(sel)))
        : Boolean(document.querySelector(sel));
      if (hit) matched.push(k);
      else if (!opt.has(k)) missingRequired.push(k);
    } catch (e) { invalid.push(`${k}: ${e.message}`); }
  }
  const total = Object.keys(selMap).length;
  return {ok: !invalid.length && !missingRequired.length, total, matched: matched.length, missingRequired, invalid};
})"""

async def _smoke_click_editor(page, mtype, global_name):
    cid = await page.eval(f"""(() => {{
      const cells = [...document.querySelectorAll('#dialog-container > .editable[id]')];
      for (const el of cells) {{
        if (el.hidden || el.classList.contains('craft-omitted')) continue;
        const card = el.querySelector(':scope > .card-in');
        if (!card || card.dataset.mtype !== {json.dumps(mtype)} || card.classList.contains('collapsed')) continue;
        const content = card.querySelector('.msg-content');
        if (!content) continue;
        const r = content.getBoundingClientRect();
        if (r.width > 4 && r.height > 4) {{ content.scrollIntoView({{block: 'center'}}); return el.id; }}
      }}
      return null;
    }})()""")
    if not cid: return True, f'skipped (no visible {mtype} cell)', True
    mounted = active = unmounted = False
    hidden_ok = True
    try:
        await page.sel_click(f'#{cid} > .card-in .msg-content')
        await page.wait_for_selector(f'#{cid} .cm-editor .cm-content', timeout=10)
        mounted = True
        active = bool(await page.eval(f"Boolean(window.{global_name}?.active?.())"))
        hidden_ok = bool(await page.eval(_FULL_EDITOR_HIDDEN_JS))
    except Exception as exc: return False, f"id={cid}, err={exc}", bool(await page.eval(_FULL_EDITOR_HIDDEN_JS))
    finally:
        try:
            await page.press('Escape')
            if await page.eval(f"Boolean(window.{global_name}?.active?.())"): await page.eval(f"window.{global_name}?.exit?.()")
            await page.wait_for_selector(f'#{cid} .cm-editor', present=False, timeout=5)
            unmounted = True
        except Exception: unmounted = False
    return bool(mounted and active and unmounted), f"id={cid}, active={active}, unmounted={unmounted}", hidden_ok

async def _smoke_native_dialog(page):
    opened = is_dialog = closed = False
    await page.press('P', mod=True, shift=True)
    try:
        await page.wait_for_selector('dialog#solveit-launcher[open]', timeout=3)
        opened = True
        is_dialog = bool(await page.eval("(() => { const d = document.getElementById('solveit-launcher'); return d?.tagName === 'DIALOG' && d.open === true; })()"))
    except Exception: opened = False
    await page.press('Escape')
    try:
        await page.wait_for_selector('dialog#solveit-launcher[open]', present=False, timeout=3)
        closed = True
    except Exception: closed = False
    return bool(opened and is_dialog and closed), f"open={opened}, dialog={is_dialog}, closed={closed}"

async def _smoke_file_tree(page):
    opened = expanded = filtered = focus_kept = clear_focus = closed = False
    filter_sel = '#solveit-file-tree-panel .solveit-tree-filter'
    await page.press('b', mod=True)
    try:
        await page.wait_for_selector('#solveit-file-tree-panel.open', timeout=3)
        await page.wait_for_selector('#solveit-file-tree .solveit-tree-row', timeout=5)
        opened = True
        folder_state = await page.eval(r"""(() => {
          const ds = [...document.querySelectorAll('#solveit-file-tree details.solveit-tree-folder')];
          if (!ds.length) return 'none';
          if (ds.some(d => d.open)) return 'open';
          return 'closed';
        })()""")
        if folder_state == 'closed':
            await page.sel_click('#solveit-file-tree details.solveit-tree-folder:not([open]) > summary.solveit-tree-row')
            await page.wait_for_selector('#solveit-file-tree details.solveit-tree-folder[open]', timeout=5)
            expanded = True
        else: expanded = True
        await page.sel_click(filter_sel)
        await page.fill_text(filter_sel, '__craft_smoke_no_match__')
        await page.wait_for("document.querySelectorAll('#solveit-file-tree li.solveit-tree-item').length === 0", timeout=3)
        filtered = True
        focus_kept = bool(await page.eval(f"document.activeElement?.matches({json.dumps(filter_sel)})"))
        await page.fill_text(filter_sel, '')
        await page.wait_for("document.querySelectorAll('#solveit-file-tree li.solveit-tree-item').length >= 1", timeout=3)
        clear_focus = bool(await page.eval(f"document.activeElement?.matches({json.dumps(filter_sel)})"))
    except Exception as exc: detail = f"open={opened}, expand={expanded}, filter={filtered}, focus={focus_kept}, clear={clear_focus}, err={exc}"
    else: detail = f"open={opened}, expand={expanded}, filter={filtered}, focus={focus_kept}, clear={clear_focus}"
    finally:
        await page.press('b', mod=True)
        try:
            await page.wait_for_selector('#solveit-file-tree-panel.open', present=False, timeout=3)
            closed = True
        except Exception: closed = False
    return bool(opened and expanded and filtered and focus_kept and clear_focus and closed), f"{detail}, closed={closed}"

Non-destructive interaction checks exercise inline code/note editing, native `<dialog>` open/Escape, file-tree filter focus preservation, and `CORE.SOLVEIT.sel` DOM contracts.

In [ ]:
#| export
async def asmoke_dialog(*, url=None, auth_file=DEFAULT_AUTH_FILE, mode='local', assets_dir=None, wait_ms=2000,
    screenshot_dir=None, headed=False, browser_name='chrome', executable_path=None, mobile=False):
    "Run non-destructive UI smoke checks against a Solveit dialog via `fastcdp`."
    checks = []
    def record(name, ok, detail=''): checks.append((name, ok, detail))

    try:
        async with open_solveit(
            url=url, mode=mode, assets_dir=assets_dir, auth_file=auth_file, wait_ms=wait_ms,
            headed=headed, browser_name=browser_name, executable_path=executable_path, mobile=mobile,
        ) as (page, intercepted, page_errors):
            record('authenticated', True, await page.eval('location.href'))
            health = await page.eval(HEALTH_PROBE_JS)
            record('dialog-container-present', bool(health['hasDialogContainer']))
            if mode == 'local':
                served = [r for r in intercepted if r.action == 'served-local']
                record('local-assets-intercepted', bool(served), f"{len(served)} files")
            if mode != 'vanilla':
                missing = [g for g in _CORE_GLOBALS if health['globals'].get(g) is None]
                record('core-craft-globals', not missing, f"missing={missing}" if missing else 'all present')
                full_ed_ok = bool(await page.eval(_FULL_EDITOR_HIDDEN_JS))

                ok, det, hid = await _smoke_click_editor(page, 'code', 'craftC2')
                full_ed_ok = full_ed_ok and hid
                record('code-cell-click-edit', ok, det)

                ok, det, hid = await _smoke_click_editor(page, 'note', 'craftMd')
                full_ed_ok = full_ed_ok and hid
                record('note-cell-click-edit', ok, det)

                ok, det = await _smoke_native_dialog(page)
                record('native-dialog-modal', ok, det)

                ok, det = await _smoke_file_tree(page)
                record('file-tree-expand-filter', ok, det)

                c_res = await page.eval(f"{_SOLVEIT_CONTRACT_JS}({json.dumps(sorted(_OPTIONAL_SOLVEIT_SELS))})")
                if 'reason' in c_res: record('solveit-dom-contracts', False, c_res['reason'])
                else:
                    det = f"{c_res['matched']}/{c_res['total']} matched"
                    if c_res['missingRequired']: det += f", missing={c_res['missingRequired']}"
                    if c_res['invalid']: det += f", invalid={c_res['invalid']}"
                    record('solveit-dom-contracts', bool(c_res['ok']), det)

                full_ed_ok = full_ed_ok and bool(await page.eval(_FULL_EDITOR_HIDDEN_JS))
                record('full-editor-hidden', full_ed_ok, '#full_editor removed & Monaco disposed' if full_ed_ok else '#full_editor or Monaco still present')
            record('no-uncaught-page-errors', not page_errors, '; '.join(page_errors[:3]))
            if screenshot_dir:
                out = Path(screenshot_dir)
                out.mkdir(parents=True, exist_ok=True)
                (out/f'smoke-{mode}.png').write_bytes(await page.screenshot())
    except PermissionError as exc: record('authenticated', False, str(exc))
    return checks

@delegates(asmoke_dialog)
def smoke_dialog(**kwargs):
    "Synchronous wrapper around `asmoke_dialog`."
    return asyncio.run(asmoke_dialog(**kwargs))

## Live `fastcdp` Bootstrap & Vanilla Mode Verification

Run with `.venv/bin/nbdev-test --flags live` against an authenticated Solveit dialog.

In [ ]:
#| live
CORE_GLOBALS = ('craftC2', 'craftMd', 'craftOmit', 'craftParts', 'craftActionMenus', 'craftCellChains', 'craftTables', 'craftNoMonaco',
    'craftInsertPreview', 'craftImageDrop', 'craftMobile', 'craftMobileBar', 'craftContext', 'craftFonts', 'craftStruct', 'craftNavRef',
    'craftHistory', 'craftTTS', 'craftBugCapture', 'solveitHotkeys', 'solveitFileTree')

async with open_solveit() as (page, intercepted, page_errors):
    assert not page_errors, f"Uncaught page errors: {page_errors}"
    health = await page.eval(HEALTH_PROBE_JS)
    assert health['hasDialogContainer']
    assert all(health['cssVars'].get(k) for k in ('--craft-prose', '--craft-code', '--craft-prose-family', '--craft-code-family'))
    missing = await page.eval(f"{json.dumps(list(CORE_GLOBALS))}.filter(n => window[n] == null)")
    assert not missing, f"Missing CRAFT globals: {missing}"
    if default_craft_mode() == 'local':
        served = {r.url for r in intercepted if r.action == 'served-local'}
        for asset in ('CRAFT.css', 'CRAFT.js', 'craft_md.js', 'hotkey_manager.js'):
            assert any(asset in u for u in served), f"Expected local interception for {asset}"

async with open_solveit(mode='vanilla') as (page, intercepted, page_errors):
    assert not page_errors, f"Uncaught page errors in vanilla mode: {page_errors}"
    assert await page.sel_exists('#dialog-container')
    assert not any(r.action == 'served-local' for r in intercepted)
    assert await page.eval("window.craftC2 === undefined && window.craftMd === undefined && window.craftMobile === undefined")